# Evaluation Agent with LLM Judges

This section evaluates the restaurant comparison agent using LLM-as-a-judge scoring. The evaluation checks whether the agent answers the user’s question, stays grounded in tool outputs, uses the appropriate tools, and gracefully rejects out-of-scope questions. The evaluation is run through MLflow so that traces and judge scores can be reviewed in the Databricks Experiments UI.

### 8.1. Set MLflow Experiment

In [0]:
import mlflow

# Set MLflow registry URI explicitly to avoid Spark config access on Serverless
mlflow.set_registry_uri("databricks-uc")

# Set MLflow experiment for final project traces and evaluation
current_user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_NAME = f"/Users/{current_user}/aai510_restaurant_agent_final"

mlflow.set_experiment(EXPERIMENT_NAME)

print("=" * 60)
print("MLflow experiment configured...")
print("=" * 60)
print(f"Experiment: {EXPERIMENT_NAME}")

### 8.2. Create Evaluation Questions

In [0]:
# Evaluation questions for LLM judge
# These are the same types of prompts tested in Section 7

EVALUATION_QUESTIONS = [
    {
        "question": "How many reviews does Parma Cucina Italiana have, and what is its average rating?",
        "expected_tools": ["business_lookup"]
    },
    {
        "question": "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?",
        "expected_tools": ["competitor_comparison"]
    },
    {
        "question": (
            "What are customers saying about the service and staff friendliness at Parma Cucina Italiana?"
        ),
        "expected_tools": ["theme_retrieval"]
    },
    {
        "question": "What is the best recipe for homemade lasagna?",
        "expected_tools": []
    },
    {
        "question": "Can you summarize reviews for a dentist in New York?",
        "expected_tools": []
    }
]

print("=" * 60)
print("Evaluation questions created...")
print("=" * 60)
print(f"Total evaluation questions: {len(EVALUATION_QUESTIONS)}")

display(EVALUATION_QUESTIONS)

### 8.3. Convert Questions to MLflow Evaluation Format

In [0]:
# Convert Section 8 evaluation questions into MLflow GenAI evaluation format
# MLflow requires expectations to be a dictionary

eval_data = []

for item in EVALUATION_QUESTIONS:
    expected_tools_text = ", ".join(item.get("expected_tools", []))

    if expected_tools_text == "":
        expected_tools_text = "none; the agent should reject the request if it is out of scope"

    eval_data.append(
        {
            "inputs": {
                "question": item["question"]
            },
            "expectations": {
                "expected_tools": expected_tools_text,
                "expected_behavior": (
                    "The agent should answer the question using the appropriate restaurant analysis tools, "
                    "stay grounded in retrieved reviews, business metadata, or tool outputs, "
                    "provide specific and useful business insight, and reject the request gracefully if it is out of scope."
                )
            }
        }
    )

print("=" * 60)
print("Evaluation dataset created...")
print("=" * 60)
print(f"Total evaluation records: {len(eval_data)}")

# Do not use display(eval_data) here because Databricks may not infer nested schemas correctly
for row in eval_data:
    print("\nQuestion:")
    print(row["inputs"]["question"])
    print("Expected tools:")
    print(row["expectations"]["expected_tools"])
    print("Expected behavior:")
    print(row["expectations"]["expected_behavior"])

### 8.4. Create Prediction Function

In [0]:
from mlflow.types.responses import ResponsesAgentRequest

def predict_fn(question: str) -> str:
    """
    Run the default restaurant agent on a single evaluation question.
    The default agent uses Llama 3.3 70B.
    """
    
    request = ResponsesAgentRequest(
        input=[{"role": "user", "content": question}]
    )
    response = AGENT.predict(request)
    
    return response

print("Prediction function created.")

In [0]:
%pip install -U -qqqq backoff databricks-openai databricks-agents "mlflow>=3.9"
dbutils.library.restartPython()

In [0]:
# Import the agent implementation and create the default agent instance
import sys
sys.path.append('/Workspace/Shared/aai-510_final_project')

import agent
import importlib
importlib.reload(agent)

# LLM endpoint for evaluation
LLAMA_3_3_70B_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

# Create agent instance
AGENT = agent.RestaurantToolCallingAgent(
    llm_endpoint=LLAMA_3_3_70B_ENDPOINT,
    tools=agent.TOOL_INFOS
)

print("=" * 60)
print("Agent loaded for evaluation")
print("=" * 60)
print(f"LLM endpoint: {LLAMA_3_3_70B_ENDPOINT}")

In [0]:
## quick test
predict_fn("How many reviews does Parma Cucina Italiana have, and what is its average rating?")

### 8.5. Create T/F LLM Judge

In [0]:
from mlflow.genai.judges import make_judge

restaurant_owner_judge = make_judge(
    name="restaurant_owner_judge",
    instructions="""
You are evaluating a restaurant-review analysis agent for a final project.

The agent's purpose is to help a small restaurant owner understand San Diego Italian restaurant performance using:
- business metadata,
- rating metrics,
- rating distributions,
- retrieved review evidence,
- competitor comparison outputs,
- and tool outputs from the agent.

User input:
{{ inputs }}

Agent response:
{{ outputs }}

Expected behavior:
{{ expectations }}

Trace:
{{ trace }}

You must evaluate the agent response using the criteria below.

Return TRUE only if all required criteria are satisfied.
Return FALSE if any required criterion is not satisfied.

Criteria:

1. Groundedness:
Is the response grounded in the retrieved reviews, business metadata, or tool outputs?
True if the response does not invent unsupported facts or make claims that cannot be traced to the available evidence.

2. Answer relevance:
Does the response answer the user’s actual question?
True if the response directly addresses the user’s request rather than giving a generic or unrelated answer.

3. Restaurant-owner usefulness:
Is the response useful for a restaurant owner?
True if the response provides practical insight, decision support, or a clear next step.

4. Specificity:
Is the response specific enough to be meaningful?
True if the response includes concrete themes, complaints, strengths, comparisons, or recommendations rather than vague statements.

5. Appropriate tool use:
Did the agent use the appropriate tool or tool sequence?
True if the tool use matches the query type, such as lookup for metadata, retrieval for review evidence, or competitor comparison for comparison questions.

6. Out-of-scope handling:
If the input was irrelevant or out of scope, did the agent reject it gracefully?
True if the agent politely refused or redirected the user to an appropriate restaurant-review analysis task.
For in-scope questions, treat this criterion as satisfied.

7. Clarity:
Is the response clear and understandable for a small business owner?
True if the response is organized, concise, and written in plain business language.

Final decision rule:
- Return TRUE if all applicable criteria are satisfied.
- Return FALSE if any applicable criterion fails.
- Return only TRUE or FALSE.
- Do not include explanations, markdown, or extra text.
""",
    model="databricks:/databricks-gpt-oss-20b",
    feedback_value_type=bool
)

print("Custom TRUE/FALSE judge created: restaurant_owner_judge")

### 8.6. Run LLM Judge Evaluation

In [0]:
import mlflow

print("=" * 60)
print("Running LLM judge evaluation...")
print("=" * 60)

eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        restaurant_owner_judge
    ]
)

print("\nEvaluation complete.")
print("Review the MLflow evaluation results in the Experiments UI.")